### Hacer chunking recursivo de la documentacion. Debes elegir los parametros más adecuados con respecto al estilo de tus documentos.

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

if API_KEY:
    print("¡API Key cargada correctamente!")
else:
    print("No se pudo encontrar la API Key. Revisa que el archivo .env esté en la ruta correcta.")

¡API Key cargada correctamente!


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# try:
#     from google.colab import userdata
#     API_KEY = userdata.get("GEMINI_API_KEY")
# except Exception:
#     from getpass import getpass
#     API_KEY = getpass("Introduce tu API key de Gemini: ")

BASE_URL =os.getenv("GOOGLE_GEMINI_BASE_URL")
MODELO = "gemini-3.5-flash-lite"

llm = ChatGoogleGenerativeAI(model=MODELO, api_key=API_KEY, base_url=BASE_URL, temperature=0)

# Prueba rápida de que el LLM responde
# print(llm.invoke("Di 'hola' en gallego, solo la palabra.").content)

/home/carolinamesamartine/ragTarea/proyecto_RAG/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Ola', 'extras': {'signature': 'EokBCoYBAWkUfRP7Ssns6fz1e/e1c2/iBERTVXaf7MO8BgNjy9j9kx/aLDyMYhp+zLZ1kCyuZhgrWTYjONTTpG9d/k5keqDowLfBWJyeIFMDFRyfpPuIJTLhxBuNihOXAYIeitTPzPkG/R4i9Plmjx5xK4XQusIU5vn0dd8jgCdD/ejPRpVzsw14Q+c='}}]


In [ ]:
import logging
from pathlib import Path
from typing import List

import docx
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

logger = logging.getLogger(__name__)

DOCS_DIR = Path.cwd() / "docs"
SUPPORTED_EXTS = {".txt", ".md", ".pdf", ".docx"}

# print(DOCS_DIR) prueba

def _meta(path: Path, **extra) -> dict:
    return {"source": str(path), "doc_key": path.name, **extra}


def load_txt_md(path: Path) -> List[Document]:
    try:
        txt = path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        txt = path.read_text(encoding="cp1252", errors="replace")
    if not txt.strip():
        return []
    return [Document(page_content=txt, metadata=_meta(path))]


def load_pdf(path: Path) -> List[Document]:
    reader = PdfReader(str(path))
    if reader.is_encrypted:
        reader.decrypt("")  # PDFs con contraseña vacía
    docs: List[Document] = []
    for i, page in enumerate(reader.pages):
        txt = page.extract_text() or ""
        if txt.strip():
            docs.append(Document(page_content=txt, metadata=_meta(path, page=i + 1)))
    return docs


def load_docx(path: Path) -> List[Document]:
    d = docx.Document(str(path))
    parts = [p.text for p in d.paragraphs if p.text.strip()]
    for table in d.tables:
        for row in table.rows:
            cells = [c.text.strip() for c in row.cells if c.text.strip()]
            if cells:
                parts.append(" | ".join(cells))
    txt = "\n".join(parts).strip()
    if not txt:
        return []
    return [Document(page_content=txt, metadata=_meta(path))]


LOADERS = {".txt": load_txt_md, ".md": load_txt_md, ".pdf": load_pdf, ".docx": load_docx}


def load_file(path: Path) -> List[Document]:
    loader = LOADERS.get(path.suffix.lower())
    if loader is None:
        return []
    try:
        return loader(path)
    except Exception as e:
        logger.warning("No se pudo leer %s: %s", path, e)
        return []


/home/carolinamesamartine/ragTarea/proyecto_RAG/notebooks/docs


In [11]:
def chunkenizar(docs_dir: Path = DOCS_DIR, chunk_size: int = 300, chunk_overlap: int = 100) -> List[Document]:
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap debe ser menor que chunk_size")

    docs_d = Path(docs_dir).resolve()
    files = sorted(
        p for p in docs_d.rglob("*")
        if p.is_file()
        and p.suffix.lower() in SUPPORTED_EXTS
        and not p.name.startswith(("~$", "."))   # ignora temporales de Word y ocultos
    )
    if not files:
        logger.warning("No hay documentos en %s (pdf/docx/txt/md).", docs_d)
        return []

    raw_docs: List[Document] = []
    for p in files:
        raw_docs.extend(load_file(p))
    if not raw_docs:
        logger.warning("No se pudo extraer texto útil de los documentos.")
        return []

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        add_start_index=True,
    )
    chunks = splitter.split_documents(raw_docs)

    for i, c in enumerate(chunks):
        pagina = f":p{c.metadata['page']}" if "page" in c.metadata else ""
        c.metadata["chunk_id"] = f"{Path(c.metadata['doc_key']).stem}{pagina}:c{i}"

    print(f"{len(files)} archivos -> {len(raw_docs)} documentos/páginas -> {len(chunks)} chunks")
    return chunks

In [ ]:
chunks300 = chunkenizar(DOCS_DIR, chunk_size=300, chunk_overlap=100)

textos = [c.page_content for c in chunks300]

# Vemos los primeros chunks y sus metadatos
for c in chunks300[:5]:
    print(c.metadata["chunk_id"], "|", c.page_content[:80].replace("\n", " "))

4 archivos -> 4 documentos/páginas -> 4 chunks
mascotas:c0 | Las mascotas más leales son los perros y los gatos
mascotas_alimentos:p1:c1 | Para   alimentar   bien   a   tu   mascota   deberás   de   darle   una   dieta 
mascotas_juguetes:c2 | Las pelotas son los juguetes por excelencia para poder entrenar bien a tu perro
mascotas_pequeñas:c3 | Las mascotas domésticas más diminutas son los ratones y los pájaros


In [ ]:
chunks30 = chunkenizar(DOCS_DIR, chunk_size=30, chunk_overlap=10) #define cuántos caracteres o tokens se repiten entre el final de un fragmento (chunk) y el inicio del siguiente

textos = [c.page_content for c in chunks30]

# Vemos los primeros chunks y sus metadatos
for c in chunks30:
    print(c.metadata["chunk_id"], "|", c.page_content[:80].replace("\n", " "))

4 archivos -> 4 documentos/páginas -> 13 chunks
mascotas:c0 | Las mascotas más leales son
mascotas:c1 | son los perros y los gatos
mascotas_alimentos:p1:c2 | Para   alimentar   bien   a
mascotas_alimentos:p1:c3 | a   tu   mascota   deberás
mascotas_alimentos:p1:c4 | deberás   de   darle   una
mascotas_alimentos:p1:c5 | una   dieta   variada
mascotas_juguetes:c6 | Las pelotas son los juguetes
mascotas_juguetes:c7 | juguetes por excelencia para
mascotas_juguetes:c8 | para poder entrenar bien a tu
mascotas_juguetes:c9 | bien a tu perro
mascotas_pequeñas:c10 | Las mascotas domésticas más
mascotas_pequeñas:c11 | más diminutas son los ratones
mascotas_pequeñas:c12 | ratones y los pájaros


In [ ]:
chunksde10 = chunkenizar(DOCS_DIR, chunk_size=10, chunk_overlap=9) #define cuántos caracteres o tokens se repiten entre el final de un fragmento (chunk) y el inicio del siguiente

textos = [c.page_content for c in chunksde10]

# Vemos los primeros chunks y sus metadatos
for c in chunksde10:
    print(c.metadata["chunk_id"], "|", c.page_content[:80].replace("\n", " "))

4 archivos -> 4 documentos/páginas -> 46 chunks
mascotas:c0 | Las
mascotas:c1 | mascotas
mascotas:c2 | más
mascotas:c3 | leales
mascotas:c4 | son los
mascotas:c5 | perros y
mascotas:c6 | y los
mascotas:c7 | los gatos
mascotas_alimentos:p1:c8 | Para
mascotas_alimentos:p1:c9 | alimentar
mascotas_alimentos:p1:c10 | bien
mascotas_alimentos:p1:c11 | bien   a
mascotas_alimentos:p1:c12 | a   tu
mascotas_alimentos:p1:c13 | a   tu
mascotas_alimentos:p1:c14 | mascota
mascotas_alimentos:p1:c15 | mascota
mascotas_alimentos:p1:c16 | deberás
mascotas_alimentos:p1:c17 | deberás
mascotas_alimentos:p1:c18 | de
mascotas_alimentos:p1:c19 | darle
mascotas_alimentos:p1:c20 | una
mascotas_alimentos:p1:c21 | dieta
mascotas_alimentos:p1:c22 | variada
mascotas_alimentos:p1:c23 | variada
mascotas_juguetes:c24 | Las
mascotas_juguetes:c25 | pelotas
mascotas_juguetes:c26 | son los
mascotas_juguetes:c27 | juguetes
mascotas_juguetes:c28 | por
mascotas_juguetes:c29 | excelenci
mascotas_juguetes:c30 | excelencia
masco